In [8]:
import numpy as np

import sys
import os


#zelf
from modules import NNCompute
from modules import configLoader
from modules import NNGeometryN
from modules.NNGeometryN import Layer
from modules import validateInputs



# --------------------------------------------------
# 1. Load config
# --------------------------------------------------

import os
    
    # load the config file
from pathlib import Path
BASE_DIR = Path.cwd()

configFile = BASE_DIR / "configParseBody.ini"
config = configLoader.load_config(configFile)



validateInputs.validateInputs(config)



print("Config loaded.")
print("Cavity type:", config.cavityType)

# --------------------------------------------------
# 2. Define a SINGLE test layer
# --------------------------------------------------

layer = Layer(
    xMin = -config.xExtent / 2,
    xMax =  config.xExtent / 2,
    yMin = -config.yExtent / 2,
    yMax =  config.yExtent / 2,
    zTop = 0,
    zBot = config.zExtent,
    vP = 3000,
    vS = 1500,
    rho = 2000
)

# classify intersection
zC = config.cavityDepth
zS = config.cavityHeight / 2 if config.cavityType == "cuboid" else config.cavityRadius

layer.intersectionType = layer.classifyIntersection(zC - zS, zC + zS)

print("Intersection type:", layer.intersectionType)

# --------------------------------------------------
# 3. Generate blocks
# --------------------------------------------------
layer.generateBlocks(
    xMin=-config.xExtent/2,
    xMax= config.xExtent/2,
    yMin=-config.yExtent/2,
    yMax= config.yExtent/2,
    zC=zC,
    zS=zS,
    domXYBounds=(
        -config.xExtent/2,
         config.xExtent/2,
        -config.yExtent/2,
         config.yExtent/2
    ))

print(f"Generated {len(layer.blocks)} blocks")

# --------------------------------------------------
# 4. Fake displacement input (FAST TEST)
# --------------------------------------------------
nFreq = 5
freqOut = np.linspace(1, 10, nFreq)

def fake_disp(z, x, y):
    # simple synthetic displacement field
    # shape: (nFreq, nPoints, 3)
    nPoints = len(x)
    disp = np.zeros((nFreq, nPoints, 3), dtype=np.complex128)

    for i in range(nFreq):
        disp[i,:,0] = np.sin(0.001 * x)
        disp[i,:,1] = np.sin(0.001 * y)
        disp[i,:,2] = np.exp(z / 1000)

    return disp

# --------------------------------------------------
# 5. Run NN computation per block
# --------------------------------------------------
I_total = np.zeros((nFreq, 3), dtype=np.complex128)

for i, block in enumerate(layer.blocks):
    print(f"Running block {i+1}/{len(layer.blocks)}")

    result = NNCompute.runVolNNComputeBlock(
        block=block,
        gridSize=200,
        freqOut=freqOut,
        cubeC=zC,

        rCavity=(config.cavityLength,
                config.cavityWidth,
                config.cavityHeight) if config.cavityType == "cuboid"
                else config.cavityRadius,

        cavity_length=config.cavityLength,
        cavity_width=config.cavityWidth,
        cavity_height=config.cavityHeight,
        cavity_type=config.cavityType,

        xSrc=0, ySrc=0,
        azSrc=[0],
        srcMeta={
            "phiSrc": np.array([0.0], dtype=float),
            "ampSrc": np.array([1.0], dtype=float)
        },
        idxFreq=None,
        fMin=None, fMax=None,
        outDispPath=None,
        splitAll=None,
        xMaxGF=None,
        fInpPath=None,
        components=None,
        vR=None,
        useSimDisp=False,
        ifRay=1,
        ifBody=0,
        nCPUDisp=1,
        nCPUNN=1,
        nChunk=5000
    )

    I_total += result

print("Done.")
print("I_total:", I_total)



Attempted to read config file: /home/jasvwild/pySeisNN1D/configParseBody.ini
Files actually read by ConfigParser: ['/home/jasvwild/pySeisNN1D/configParseBody.ini']
Current working directory: /home/jasvwild/pySeisNN1D
Sections found: ['model', 'frequency', 'grid', 'geometry', 'simSource', 'paths', 'simFlags', 'compute']
All configuration and model checks passed.
Config loaded.
Cavity type: cuboid
Intersection type: contains
Generated 11 blocks
Running block 1/11
Running block 2/11
Running block 3/11
Running block 4/11
Running block 5/11
Running block 6/11
Uniform spacing
[WARNING] All points removed at z=-240.50 in cavity block — skipping this slice.
[WARNING] All points removed at z=-241.50 in cavity block — skipping this slice.
[WARNING] All points removed at z=-242.50 in cavity block — skipping this slice.
[WARNING] All points removed at z=-243.50 in cavity block — skipping this slice.
[WARNING] All points removed at z=-244.50 in cavity block — skipping this slice.
[WARNING] All poin